# Lesson 8: Forces and integration

**Why three orbiting bodies rarely repeat** · Advanced · from [Hypnotic Patterns](https://hypnoticpatterns.doner.cloud)

Two bodies under gravity circle each other on the same ellipse for ever. Add a third of equal mass and almost every start ends in a tangle, usually with one body thrown out. Yet a rare few starts close on themselves and repeat, like the figure-eight.

Watch the films and play with the live demo: [Three-body orbits](https://hypnoticpatterns.doner.cloud/pattern/three-body-orbits). The same lesson with a quiz: [https://hypnoticpatterns.doner.cloud/learn/three-body-orbits](https://hypnoticpatterns.doner.cloud/learn/three-body-orbits).

## What you will learn

- Newton’s gravity as vectors
- nested loops over pairs of bodies
- the leapfrog method of integration
- a 3-D array: step × body × coordinate
- checking a simulation by what must not change

## The script

Run the cell below (Shift + Enter). numpy and matplotlib are already in Colab.

In [ ]:
"""Three bodies: the figure-eight orbit, from the starting values computed by Carles Simo
(in Chenciner and Montgomery, 2000). G = 1, three masses of 1.
Run: python three-body-orbits.py   (needs: pip install numpy matplotlib)
"""
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")

x1 = np.array([0.97000436, -0.24308753])
V = np.array([-0.93240737, -0.86473146])  # the velocity of body 3
pos = np.array([x1, -x1, [0.0, 0.0]])
vel = np.array([-V / 2, -V / 2, V])
period, dt = 6.32591398, 0.0005
turn = 0  # degrees to turn the first velocity by (the film breaks the eight with 15)

a = np.radians(turn)
vel[0] = [vel[0, 0] * np.cos(a) - vel[0, 1] * np.sin(a), vel[0, 0] * np.sin(a) + vel[0, 1] * np.cos(a)]


def pull(p):
    """Each body's acceleration: the sum of the other two's gravity, G*m / r**2 towards each"""
    acc = np.zeros_like(p)
    for i in range(3):
        for j in range(3):
            if i != j:
                d = p[j] - p[i]
                acc[i] += d / np.linalg.norm(d) ** 3
    return acc


track = []
acc = pull(pos)
for _ in range(int(3 * period / dt)):  # three periods, leapfrog steps (they keep the energy well)
    vel += acc * dt / 2
    pos += vel * dt
    acc = pull(pos)
    vel += acc * dt / 2
    track.append(pos.copy())
track = np.array(track)

colours = ["#5eead4", "#f0abfc", "#f5c76b"]
plt.figure(figsize=(7, 4))
for body in range(3):  # all three run along the same track, a third of a period apart
    plt.plot(track[:, body, 0], track[:, body, 1], color=colours[body], linewidth=1)
plt.scatter(track[-1, :, 0], track[-1, :, 1], c=colours, s=80, zorder=3)
plt.axis("equal")
plt.axis("off")
plt.show()

# Try: turn = 15 -> the three tracks no longer close; the eight falls apart

## Line by line

1. `def pull(p):`  
   The acceleration of each body, from the gravity of the other two.
2. `acc[i] += d / np.linalg.norm(d) ** 3`  
   Gravity G·m / r² along the direction d / r together make d / r³; here G and the masses are 1.
3. `vel += acc * dt / 2`  
   Leapfrog: half a step of velocity, a whole step of position, then the other half step with the new forces.
4. `pos += vel * dt`  
   The positions move with the half-updated velocities.
5. `track = np.array(track)`  
   All the steps as one array: track[step, body, x or y].
6. `plt.scatter(track[-1, :, 0], track[-1, :, 1], c=colours, s=80, zorder=3)`  
   The three bodies where they ended, on one shared track, a third of a period apart.

## Try this

Set turn = 15, as in the film: the three tracks no longer close.

## Exercises

Write your code in the empty cells. The hints and answers are at the end of the notebook.

1. Where is the centre of mass at the end?
2. Set turn = 15, as in the film. What happens, and where does the centre of mass go now?
3. Make dt ten times bigger, 0.005. Does the eight still close?

**Exercise 1.** Where is the centre of mass at the end?

In [ ]:
# your code here

**Exercise 2.** Set turn = 15, as in the film. What happens, and where does the centre of mass go now?

In [ ]:
# your code here

**Exercise 3.** Make dt ten times bigger, 0.005. Does the eight still close?

In [ ]:
# your code here

## Why it works

Each body pulls the other two with Newton’s gravity, F = G·m₁·m₂ ÷ r². For two bodies the equations have an exact answer, the ellipse. For three there is no general formula: the motion has to be worked out step by step, and it is chaotic, so the tiniest change at the start grows until the orbits look nothing alike.

A periodic solution is a start that comes back exactly to itself after one period. The figure-eight, three equal masses chasing each other round one track, was found by computer by Cris Moore in 1993 and proved to exist by Alain Chenciner and Richard Montgomery in 2000.

Computer studies show the figure-eight survives small changes of the masses and the start, which most periodic orbits do not; the film’s 15° turn of one velocity is far bigger, and the ballet falls apart. In 2013 Milovan Šuvakov and Veljko Dmitrašinović found 13 new families of periodic orbits by a computer search, among them the “moth” and the “yin-yang”.

With zero total momentum the centre of mass never moves, and the total energy stays the same. That is how a simulation is checked: a good integrator keeps both very nearly constant over many periods.

`aᵢ = G · Σⱼ≠ᵢ mⱼ · (rⱼ − rᵢ) ÷ |rⱼ − rᵢ|³      for the figure-eight: three equal masses, zero angular momentum`

## Hints and answers

1. *Hint:* track[-1].mean(axis=0).  
   *Answer:* At (0, 0) to within about 10⁻¹³: the total momentum is zero, so it never moves.
2. *Hint:* Compare the start with the positions after one period.  
   *Answer:* The tracks no longer close: after one period the three bodies are, together, about 2 units away from where they began. Turning one velocity also gives the system some momentum, so the centre of mass drifts, to about (−0.81, 0.67) at the end.
3. *Hint:* Measure how far the bodies are from the start after one period.  
   *Answer:* Yes, but less exactly: after one period the three are, together, about 0.006 away from the start, against about 0.0001 with dt = 0.0005.

## Sources

- [Three-body problem (Wikipedia)](https://en.wikipedia.org/wiki/Three-body_problem)
- [A. Chenciner, R. Montgomery, “A remarkable periodic solution of the three-body problem in the case of equal masses”, Annals of Mathematics 152 (2000)](https://doi.org/10.2307/2661357)
- [M. Šuvakov, V. Dmitrašinović, “Three classes of Newtonian three-body planar periodic orbits”, Physical Review Letters 110 (2013)](https://doi.org/10.1103/PhysRevLett.110.114301)

Free to use for learning and in class. Made for the YouTube channel [Hypnotic Patterns](https://www.youtube.com/@HypnoticPatternsAudio).